# Notebook 02 — Hand Asymmetry Analysis
**The asymmetry score is the primary clinical recovery metric in this dataset.**

| Audience | What they get |
|---|---|
| **User / Patient** | Clear picture of which hand is struggling and by how much |
| **Doctor / Clinician** | Baseline asymmetry + shape-specific weakness for rehabilitation planning |
| **App maker** | Which shapes to prioritise for the impaired hand; adaptive difficulty signals |


In [1]:
DATA_DIR     = '.'
OUT_DIR      = 'outputs'
GROUP_LABEL  = 'Group A'
ROLLING_WIN  = 5
IMPAIRED_HAND = 'Right'   # set to 'Left' or 'Right' based on patient's impaired side
import os, json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

# ── Make sure output folder exists BEFORE anything tries to write to it ──
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, os.path.dirname(os.path.abspath('hci_utils.py')))
from hci_utils import (load_board_tries, load_bd_sessions, load_piano_sessions,
                        load_piano_movements, compare_groups, save_fig,
                        SHAPE_ORDER, HAND_COLORS, GROUP_COLORS)

df = load_board_tries(DATA_DIR)
print(f"Loaded {len(df)} tries")
print(df['hand'].value_counts().to_string())


KeyError: 'startedAt'

## 1 · Per-hand summary

In [ ]:
hand_summary = (
    df.groupby('hand')
    .agg(total_tries   = ('_id',       'count'),
         success_count = ('completed', 'sum'),
         avg_accuracy  = ('accuracy',  'mean'),
         std_accuracy  = ('accuracy',  'std'))
    .reset_index()
)
hand_summary['success_rate'] = hand_summary['success_count'] / hand_summary['total_tries'] * 100
hand_summary['std_accuracy']  = hand_summary['std_accuracy'].fillna(0)
print(hand_summary.to_string(index=False))

# Overall asymmetry score
left_row  = hand_summary[hand_summary['hand'] == 'Left']
right_row = hand_summary[hand_summary['hand'] == 'Right']
if len(left_row) and len(right_row):
    asym = float(left_row['success_rate'].values[0]) - float(right_row['success_rate'].values[0])
    print(f'\nOverall asymmetry score: {asym:+.1f} pp  (positive = Left is better)')


## 2 · Per hand × shape matrix

In [ ]:
hxs = (
    df.groupby(['hand', 'shapeType'])
    .agg(tries=('_id','count'), successes=('completed','sum'), avg_acc=('accuracy','mean'))
    .reset_index()
)
hxs['success_rate'] = hxs['successes'] / hxs['tries'] * 100
os.makedirs(OUT_DIR, exist_ok=True)
hxs.to_csv(os.path.join(OUT_DIR, 'hand_asymmetry_summary.csv'), index=False)
print('hand_asymmetry_summary.csv saved')
print(hxs.to_string(index=False))


## 3 · Fig 02a — Grouped bar: success rate per shape per hand

In [ ]:
shapes_present = [s for s in SHAPE_ORDER if s in df['shapeType'].values]
hands = sorted(df['hand'].unique())
x = np.arange(len(shapes_present))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
for i, hand in enumerate(hands):
    vals = []
    for s in shapes_present:
        row = hxs[(hxs['hand'] == hand) & (hxs['shapeType'] == s)]
        vals.append(float(row['success_rate'].values[0]) if len(row) else 0)
    ax.bar(x + i * width - width / 2, vals, width,
           label=f'{hand} hand',
           color=HAND_COLORS.get(hand, '#888'), alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(shapes_present, rotation=20, ha='right')
ax.set_ylabel('Success rate (%)')
ax.set_ylim(0, 115)
ax.set_title(GROUP_LABEL + ' — Success rate: Left vs Right hand per shape', fontweight='bold')
ax.legend()
ax.axhline(100, color='gray', ls=':', lw=1)
plt.tight_layout()
save_fig(fig, 'fig02a_hand_success_rate.png', OUT_DIR)
plt.show()

print('\n=== CLINICIAN SIGNAL ===')
print('Bars where impaired hand is significantly lower = shape-specific motor deficit.')
print('Use these shapes as targeted therapy exercises.')
print('\n=== APP-MAKER SIGNAL ===')
weakest = hxs[hxs['hand'] == IMPAIRED_HAND].nsmallest(3, 'success_rate')[['shapeType','success_rate']]
print('Top 3 shapes to prioritise for impaired hand:')
print(weakest.to_string(index=False))


## 4 · Fig 02b — Box plot: accuracy distribution by hand

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
data_l = df[df['hand'] == 'Left']['accuracy'].values
data_r = df[df['hand'] == 'Right']['accuracy'].values
plot_data = [d for d in [data_l, data_r] if len(d)]
plot_lbls = [h + ' hand' for h in ['Left', 'Right'] if len(df[df['hand'] == h])]

bp = ax.boxplot(plot_data, labels=plot_lbls, patch_artist=True,
                medianprops=dict(color='black', lw=2))
for patch, lbl in zip(bp['boxes'], ['Left', 'Right']):
    patch.set_facecolor(HAND_COLORS.get(lbl, '#888'))
    patch.set_alpha(0.6)
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110)
ax.set_title(GROUP_LABEL + ' — Accuracy distribution by hand', fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig02b_hand_accuracy_boxplot.png', OUT_DIR)
plt.show()

print('\n=== CLINICIAN SIGNAL ===')
print('Wider box for impaired hand = more inconsistent performance = motor instability.')
print('Narrowing of the impaired hand box over time = a key recovery indicator.')


## 5 · Fig 02c — Asymmetry score per shape (diverging bar)

In [ ]:
asym_rows = []
for s in shapes_present:
    l_row = hxs[(hxs['hand'] == 'Left')  & (hxs['shapeType'] == s)]
    r_row = hxs[(hxs['hand'] == 'Right') & (hxs['shapeType'] == s)]
    l = float(l_row['success_rate'].values[0]) if len(l_row) else None
    r = float(r_row['success_rate'].values[0]) if len(r_row) else None
    if l is not None and r is not None:
        asym_rows.append({'shape': s, 'asymmetry': l - r})
asym = pd.DataFrame(asym_rows)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2a6e4f' if v >= 0 else '#c84b2f' for v in asym['asymmetry']]
ax.barh(asym['shape'], asym['asymmetry'], color=colors, height=0.5)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('Left success rate - Right success rate (pp)
(green = left better, red = right better)')
ax.set_title(GROUP_LABEL + ' — Hand asymmetry score per shape', fontweight='bold')
plt.tight_layout()
save_fig(fig, 'fig02c_asymmetry_score_per_shape.png', OUT_DIR)
plt.show()

print('\n=== CLINICIAN SIGNAL ===')
worst_asym = asym.reindex(asym['asymmetry'].abs().sort_values(ascending=False).index).iloc[0]
print(f"Highest asymmetry: {worst_asym['shape']} ({worst_asym['asymmetry']:+.1f} pp)")
print('This shape most clearly distinguishes impaired vs non-impaired hand.')
print('Use it as the primary tracking metric across therapy sessions.')

print('\n=== USER FEEDBACK ===')
print('Bars pointing right = your right hand needs more work on that shape.')
print('Goal over therapy: all bars should move toward zero.')


## 6 · Fig 02d — Accuracy over time by hand

In [ ]:
df_s = df.sort_values('startedAt').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 4))
for hand in sorted(df_s['hand'].unique()):
    sub = df_s[df_s['hand'] == hand].copy().reset_index(drop=True)
    sub['rolling'] = sub['accuracy'].rolling(ROLLING_WIN, min_periods=1).mean()
    ax.plot(range(len(sub)), sub['rolling'],
            label=f'{hand} hand', color=HAND_COLORS.get(hand, '#888'), lw=2)

ax.set_xlabel(f'Try number (chronological, rolling window={ROLLING_WIN})')
ax.set_ylabel('Accuracy (%)')
ax.set_title(GROUP_LABEL + ' — Accuracy over time by hand', fontweight='bold')
ax.legend()
ax.set_ylim(0, 110)
plt.tight_layout()
save_fig(fig, 'fig02d_hand_accuracy_over_time.png', OUT_DIR)
plt.show()

print('\n=== CLINICIAN SIGNAL ===')
print('Gap between lines = current impairment level.')
print('Converging lines over therapy sessions = measurable motor recovery.')
print('\n=== USER FEEDBACK ===')
print('Two lines trending upward = both hands improving.')
print('The gap between them closing = great sign of recovery!')
